# Gaussian Mixture Model для прогноза перспективных зон

Идея ноутбука: использовать **GMM как метод обучения без учителя**.  
Модель обучается на всех ячейках сетки по геологическим признакам и выделяет скрытые типы территории.  
Известные проявления используются **не как обучающие метки**, а для интерпретации компонент и проверки результата.

In [ ]:
# ============================================================
# 0. НАСТРОЙКИ
# ============================================================

from pathlib import Path
import os
import json
import math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from shapely.geometry import box
from shapely.ops import unary_union

from sklearn.preprocessing import QuantileTransformer, StandardScaler
from sklearn.mixture import GaussianMixture

try:
    from scipy.ndimage import gaussian_filter, label as cc_label
    SCIPY_OK = True
except Exception:
    SCIPY_OK = False

# Главная папка проекта
BASE_DIR = Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз")
SHP_DIR = BASE_DIR / "shp_dbf"
OUT_DIR = BASE_DIR / "gmm_unsupervised_result"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Размер ячейки сетки в метрах
CELL_SIZE = 500

# Если у файлов нет CRS, используем рабочий CRS проекта
DEFAULT_CRS = "EPSG:32649"

# Основные слои
LAYER_MAP = {
    "mask": "svita_new",
    "facies": "fasii",
    "paleo": "gr_dol_vp_poly",
    "struct": "kory",
    "magm": "dayki_buf",
    "tect1": "glub_raz_nw",
    "tect2": "glub_r_nw",
}

# Масштабы перевода расстояний в proximity-признаки
DIST_SCALES = {
    "facies": 1000,
    "paleo": 1000,
    "struct": 900,
    "magm": 800,
    "tect1": 800,
    "tect2": 800,
}

# Слои с известными проявлениями / поисковыми признаками.
# Если таких файлов нет, ноутбук попробует найти point-слои автоматически.
EVIDENCE_LAYER_NAMES = [
    "result",
    "геохимическое_опробование",
]

# GMM: какие числа компонент попробовать
N_COMPONENTS_LIST = [3, 4, 5, 6, 7, 8, 10, 12]
COVARIANCE_TYPES = ["full", "diag"]
RANDOM_STATE = 42

# Итоговые top-зоны: 0.992 = примерно верхние 0.8% территории
TOP_ZONE_QUANTILE = 0.992
SMOOTH_SIGMA = 0.9

PNG_PATH = OUT_DIR / "gmm_forecast_map.png"
METRICS_PATH = OUT_DIR / "gmm_metrics.json"
CLUSTER_TABLE_PATH = OUT_DIR / "gmm_cluster_table.csv"

print("BASE_DIR:", BASE_DIR)
print("SHP_DIR:", SHP_DIR)
print("OUT_DIR:", OUT_DIR)

In [ ]:
# ============================================================
# 1. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ============================================================

def normalize01(x):
    x = np.asarray(x, dtype=float)
    mn = np.nanmin(x)
    mx = np.nanmax(x)
    if not np.isfinite(mn) or not np.isfinite(mx) or mx - mn < 1e-12:
        return np.zeros_like(x, dtype=float)
    return (x - mn) / (mx - mn)


def safe_name(s):
    return str(s).lower().replace("ё", "е").replace(" ", "_")


def list_vector_files(folder):
    exts = [".shp", ".gpkg", ".geojson"]
    files = []
    for ext in exts:
        files.extend(folder.glob(f"*{ext}"))
    return sorted(files)


def find_vector_file(folder, layer_name):
    """Ищет файл по имени слоя без учета регистра и расширения."""
    target = safe_name(layer_name)
    files = list_vector_files(folder)
    for f in files:
        stem = safe_name(f.stem)
        if stem == target:
            return f
    for f in files:
        stem = safe_name(f.stem)
        if target in stem or stem in target:
            return f
    return None


def read_pj4_if_exists(path):
    """Некоторые старые наборы имеют projection в *_shp.pj4."""
    candidates = [
        path.with_suffix(".prj"),
        path.with_name(path.stem + "_shp.pj4"),
        path.with_suffix(".pj4"),
    ]
    for p in candidates:
        if p.exists():
            txt = p.read_text(errors="ignore").strip()
            if txt:
                return txt
    return None


def repair_geometries(gdf):
    if gdf is None or len(gdf) == 0:
        return gdf
    gdf = gdf.copy()
    before = len(gdf)
    gdf = gdf[gdf.geometry.notna()]
    gdf = gdf[~gdf.geometry.is_empty]
    if len(gdf) == 0:
        return gdf
    try:
        # Shapely 2.x
        from shapely.validation import make_valid
        fixed = gdf.geometry.apply(make_valid)
    except Exception:
        try:
            fixed = gdf.geometry.buffer(0)
        except Exception:
            fixed = gdf.geometry
    gdf.geometry = fixed
    gdf = gdf[gdf.geometry.notna()]
    gdf = gdf[~gdf.geometry.is_empty]
    gdf = gdf.reset_index(drop=True)
    if len(gdf) == 0:
        print("Предупреждение: после repair_geometries слой стал пустым. До очистки было:", before)
    return gdf


def read_layer(folder, layer_name, target_crs=None, required=True):
    path = find_vector_file(folder, layer_name)
    if path is None:
        if required:
            raise FileNotFoundError(f"Не найден слой '{layer_name}' в папке {folder}")
        return None
    gdf = gpd.read_file(path)
    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()

    if gdf.crs is None:
        pj = read_pj4_if_exists(path)
        if pj:
            try:
                gdf = gdf.set_crs(pj, allow_override=True)
            except Exception:
                gdf = gdf.set_crs(DEFAULT_CRS, allow_override=True)
        else:
            gdf = gdf.set_crs(DEFAULT_CRS, allow_override=True)

    if target_crs is not None and gdf.crs != target_crs:
        gdf = gdf.to_crs(target_crs)

    # Для mask делаем осторожно: если repair сделал пусто, оставим исходный слой
    fixed = repair_geometries(gdf)
    if fixed is not None and len(fixed) > 0:
        gdf = fixed

    print(f"{layer_name:25s} | {len(gdf):6d} объектов | CRS: {gdf.crs}")
    return gdf


def finite_bounds(gdf, name="layer"):
    bounds = np.asarray(gdf.total_bounds, dtype=float)
    if len(bounds) != 4 or not np.all(np.isfinite(bounds)):
        raise ValueError(f"Некорректные total_bounds у {name}: {bounds}")
    minx, miny, maxx, maxy = bounds
    if maxx <= minx or maxy <= miny:
        raise ValueError(f"Некорректные размеры bounds у {name}: {bounds}")
    return minx, miny, maxx, maxy


def safe_unary_union(gdf):
    if gdf is None or len(gdf) == 0:
        return None
    try:
        return unary_union(list(gdf.geometry))
    except Exception:
        return gdf.geometry.unary_union


def proximity_to_layer(points_gdf, layer_gdf, scale):
    if layer_gdf is None or len(layer_gdf) == 0:
        return np.zeros(len(points_gdf), dtype=float), np.full(len(points_gdf), np.nan)
    geom_union = safe_unary_union(layer_gdf)
    if geom_union is None or geom_union.is_empty:
        return np.zeros(len(points_gdf), dtype=float), np.full(len(points_gdf), np.nan)
    d = points_gdf.geometry.distance(geom_union).astype(float).values
    prox = 1.0 / (1.0 + d / float(scale))
    return prox, d


def intersects_layer(cells_gdf, layer_gdf):
    if layer_gdf is None or len(layer_gdf) == 0:
        return np.zeros(len(cells_gdf), dtype=int)
    geom_union = safe_unary_union(layer_gdf)
    if geom_union is None or geom_union.is_empty:
        return np.zeros(len(cells_gdf), dtype=int)
    return cells_gdf.geometry.intersects(geom_union).astype(int).values

In [ ]:
# ============================================================
# 2. ЗАГРУЗКА СЛОЕВ
# ============================================================

if not SHP_DIR.exists():
    raise FileNotFoundError(f"Папка с SHP не найдена: {SHP_DIR}")

print("Найденные векторные файлы:")
for f in list_vector_files(SHP_DIR):
    print(" -", f.name)

print("\nЗагрузка основных слоев:")
mask = read_layer(SHP_DIR, LAYER_MAP["mask"], target_crs=None, required=True)
if mask.crs is None:
    mask = mask.set_crs(DEFAULT_CRS, allow_override=True)
mask_crs = mask.crs

layers = {"mask": mask}
for key, lname in LAYER_MAP.items():
    if key == "mask":
        continue
    layers[key] = read_layer(SHP_DIR, lname, target_crs=mask_crs, required=False)

print("\nCRS проекта:", mask_crs)

In [ ]:
# ============================================================
# 3. ПОСТРОЕНИЕ СЕТКИ 500×500 М
# ============================================================

def build_grid(mask_gdf, cell_size=500):
    mask_union = safe_unary_union(mask_gdf)
    minx, miny, maxx, maxy = finite_bounds(mask_gdf, "mask")

    xs = np.arange(minx, maxx + cell_size, cell_size)
    ys = np.arange(miny, maxy + cell_size, cell_size)

    cells = []
    rows = []
    cols = []
    cell_id = []
    k = 0
    for r, y in enumerate(ys[:-1]):
        for c, x in enumerate(xs[:-1]):
            geom = box(x, y, x + cell_size, y + cell_size)
            if geom.intersects(mask_union):
                inter = geom.intersection(mask_union)
                if not inter.is_empty:
                    cells.append(geom)
                    rows.append(r)
                    cols.append(c)
                    cell_id.append(k)
                    k += 1

    grid = gpd.GeoDataFrame(
        {"cell_id": cell_id, "row": rows, "col": cols},
        geometry=cells,
        crs=mask_gdf.crs,
    )
    grid["cx"] = grid.geometry.centroid.x
    grid["cy"] = grid.geometry.centroid.y
    return grid


grid = build_grid(mask, CELL_SIZE)
centroids = gpd.GeoDataFrame(grid[["cell_id", "row", "col"]].copy(), geometry=grid.geometry.centroid, crs=grid.crs)

print("Ячеек в сетке:", len(grid))
print("rows, cols:", grid["row"].max() + 1, grid["col"].max() + 1)
grid.head()

In [ ]:
# ============================================================
# 4. ФОРМИРОВАНИЕ ГЕОЛОГИЧЕСКИХ ПРИЗНАКОВ
# ============================================================

# Proximity-признаки: чем ближе к фактору, тем значение выше
for key in ["facies", "paleo", "struct", "magm", "tect1", "tect2"]:
    prox, dist = proximity_to_layer(centroids, layers.get(key), DIST_SCALES[key])
    grid[f"prox_{key}"] = prox
    grid[f"dist_{key}"] = dist
    grid[f"rank_{key}"] = pd.Series(prox).rank(pct=True).values

# Бинарные признаки: ячейка пересекает фактор или нет
for key in ["facies", "paleo", "struct", "magm", "tect1", "tect2"]:
    grid[f"int_{key}"] = intersects_layer(grid, layers.get(key))

# Объединенная тектоника
if "prox_tect1" in grid.columns and "prox_tect2" in grid.columns:
    grid["prox_tect"] = np.maximum(grid["prox_tect1"].values, grid["prox_tect2"].values)
    grid["rank_tect"] = pd.Series(grid["prox_tect"]).rank(pct=True).values
    grid["int_tect"] = np.maximum(grid["int_tect1"].values, grid["int_tect2"].values)
else:
    grid["prox_tect"] = 0.0
    grid["rank_tect"] = 0.0
    grid["int_tect"] = 0

# Количество активных факторов в ячейке
int_cols = ["int_facies", "int_paleo", "int_struct", "int_magm", "int_tect"]
grid["active_factor_count"] = grid[int_cols].sum(axis=1)
grid["active_factor_norm"] = normalize01(grid["active_factor_count"].values)

# Узловые признаки: совместное проявление факторов
# Важно: модель должна видеть не одиночный фактор, а сочетание условий.
grid["node_tect_magm"] = grid["prox_tect"] * grid["prox_magm"]
grid["node_tect_struct"] = grid["prox_tect"] * grid["prox_struct"]
grid["node_tect_paleo"] = grid["prox_tect"] * grid["prox_paleo"]
grid["node_magm_struct"] = grid["prox_magm"] * grid["prox_struct"]
grid["node_paleo_struct"] = grid["prox_paleo"] * grid["prox_struct"]
grid["node_facies_paleo"] = grid["prox_facies"] * grid["prox_paleo"]

# Бинарные пересечения факторов
for a, b in [("tect", "magm"), ("tect", "struct"), ("tect", "paleo"), ("magm", "struct"), ("paleo", "struct")]:
    grid[f"bin_{a}_{b}"] = (grid[f"int_{a}"] * grid[f"int_{b}"]).astype(int)

# Обобщенная сила узла
node_raw = (
    1.35 * grid["node_tect_magm"] +
    1.25 * grid["node_tect_struct"] +
    1.10 * grid["node_tect_paleo"] +
    0.85 * grid["node_magm_struct"] +
    0.75 * grid["node_paleo_struct"] +
    0.55 * grid["node_facies_paleo"] +
    0.45 * grid["active_factor_norm"]
)
grid["node_strength"] = normalize01(node_raw.values)

# Итоговый набор признаков для GMM
FEATURE_COLS = [
    "prox_facies", "prox_paleo", "prox_struct", "prox_magm", "prox_tect",
    "rank_facies", "rank_paleo", "rank_struct", "rank_magm", "rank_tect",
    "active_factor_norm",
    "node_tect_magm", "node_tect_struct", "node_tect_paleo",
    "node_magm_struct", "node_paleo_struct", "node_facies_paleo",
    "node_strength",
]

print("Признаков для GMM:", len(FEATURE_COLS))
print(FEATURE_COLS)

grid[FEATURE_COLS].describe().T

In [ ]:
# ============================================================
# 5. ЗАГРУЗКА ИЗВЕСТНЫХ ПРОЯВЛЕНИЙ / EVIDENCE POINTS
# ============================================================

def geometry_is_point_like(gdf):
    if gdf is None or len(gdf) == 0:
        return False
    geom_types = set(gdf.geometry.geom_type.dropna().unique())
    return any(t in geom_types for t in ["Point", "MultiPoint"])


def explode_points(gdf):
    if gdf is None or len(gdf) == 0:
        return gdf
    try:
        gdf = gdf.explode(index_parts=False).reset_index(drop=True)
    except Exception:
        gdf = gdf.explode().reset_index(drop=True)
    gdf = gdf[gdf.geometry.geom_type.isin(["Point", "MultiPoint"])]
    try:
        gdf = gdf.explode(index_parts=False).reset_index(drop=True)
    except Exception:
        pass
    return gdf


def load_evidence_points(folder, target_crs):
    pieces = []

    # Сначала пробуем явно заданные имена
    for lname in EVIDENCE_LAYER_NAMES:
        gdf = read_layer(folder, lname, target_crs=target_crs, required=False)
        if gdf is not None and len(gdf) > 0 and geometry_is_point_like(gdf):
            gdf = explode_points(gdf)
            gdf["evidence_source"] = lname
            pieces.append(gdf[["evidence_source", "geometry"]])

    # Если ничего не нашли, попробуем взять все point-слои
    if len(pieces) == 0:
        print("Явные evidence-слои не найдены. Пробую найти point-слои автоматически...")
        for f in list_vector_files(folder):
            try:
                gdf = gpd.read_file(f)
                if gdf.crs is None:
                    gdf = gdf.set_crs(DEFAULT_CRS, allow_override=True)
                if gdf.crs != target_crs:
                    gdf = gdf.to_crs(target_crs)
                if geometry_is_point_like(gdf):
                    gdf = explode_points(gdf)
                    if len(gdf) > 0:
                        gdf["evidence_source"] = f.stem
                        pieces.append(gdf[["evidence_source", "geometry"]])
                        print("  point-layer:", f.name, "n=", len(gdf))
            except Exception as e:
                print("  не удалось прочитать", f.name, e)

    if len(pieces) == 0:
        return gpd.GeoDataFrame({"evidence_source": []}, geometry=[], crs=target_crs)

    pts = pd.concat(pieces, ignore_index=True)
    pts = gpd.GeoDataFrame(pts, geometry="geometry", crs=target_crs)
    pts = pts[pts.geometry.notna()]
    pts = pts[~pts.geometry.is_empty]
    return pts.reset_index(drop=True)


evidence = load_evidence_points(SHP_DIR, grid.crs)
print("Evidence points:", len(evidence))
if len(evidence) > 0:
    display(evidence["evidence_source"].value_counts())

# Привязка точек к ячейкам сетки
grid["evidence_cell"] = 0
if len(evidence) > 0:
    joined = gpd.sjoin(
        evidence,
        grid[["cell_id", "geometry"]],
        how="inner",
        predicate="within"
    )
    evidence_cell_ids = sorted(joined["cell_id"].unique())
    grid.loc[grid["cell_id"].isin(evidence_cell_ids), "evidence_cell"] = 1
    print("Evidence cells:", int(grid["evidence_cell"].sum()))
else:
    print("Evidence points не найдены. Оценка будет только по node_strength.")

In [ ]:
# ============================================================
# 6. ПОДГОТОВКА МАТРИЦЫ ПРИЗНАКОВ
# ============================================================

X_df = grid[FEATURE_COLS].copy()
X_df = X_df.replace([np.inf, -np.inf], np.nan).fillna(0.0)

# Для GMM полезно сделать распределения признаков более близкими к нормальным.
# QuantileTransformer не использует метки, это только преобразование X.
n_quantiles = min(1000, len(X_df))
qt = QuantileTransformer(
    n_quantiles=n_quantiles,
    output_distribution="normal",
    random_state=RANDOM_STATE
)
scaler = StandardScaler()

X_q = qt.fit_transform(X_df.values)
X = scaler.fit_transform(X_q)

print("X shape:", X.shape)
print("NaN in X:", np.isnan(X).sum())

In [ ]:
# ============================================================
# 7. ОБУЧЕНИЕ GMM И ВЫБОР ЛУЧШЕЙ КОНФИГУРАЦИИ
# ============================================================

def cluster_statistics(grid_local, labels, proba, model_name=""):
    df = pd.DataFrame({
        "cluster": labels,
        "evidence_cell": grid_local["evidence_cell"].values,
        "node_strength": grid_local["node_strength"].values,
    })
    total_n = len(df)
    total_e = max(df["evidence_cell"].sum(), 1)
    global_e_rate = df["evidence_cell"].mean() if df["evidence_cell"].sum() > 0 else 0.0

    rows = []
    for cl in sorted(df["cluster"].unique()):
        part = df[df["cluster"] == cl]
        n = len(part)
        e = int(part["evidence_cell"].sum())
        e_rate = e / n if n > 0 else 0.0
        lift = e_rate / global_e_rate if global_e_rate > 0 else 0.0
        rows.append({
            "model": model_name,
            "cluster": int(cl),
            "n_cells": int(n),
            "area_share": n / total_n,
            "evidence_cells": e,
            "evidence_share_inside_cluster": e_rate,
            "evidence_capture": e / total_e,
            "evidence_lift": lift,
            "node_mean": float(part["node_strength"].mean()),
            "node_p90": float(part["node_strength"].quantile(0.90)),
        })
    stat = pd.DataFrame(rows)

    # Нормализованная оценка компоненты.
    # Если evidence есть, оно помогает интерпретировать компоненту.
    # Если evidence нет, остаётся только node_strength.
    if stat["evidence_cells"].sum() > 0:
        lift_score = normalize01(np.log1p(stat["evidence_lift"].values))
        capture_score = normalize01(stat["evidence_capture"].values)
        node_score = normalize01(stat["node_mean"].values)
        size_penalty = np.clip((stat["area_share"].values - 0.35) / 0.65, 0, 1)
        stat["cluster_priority"] = (
            0.45 * lift_score +
            0.25 * capture_score +
            0.30 * node_score -
            0.15 * size_penalty
        )
    else:
        stat["cluster_priority"] = normalize01(stat["node_mean"].values)

    stat["cluster_priority"] = normalize01(stat["cluster_priority"].values)
    return stat


def prospectivity_from_gmm(grid_local, labels, proba, stat):
    # Каждая компонента получает вес priority.
    priorities = stat.set_index("cluster")["cluster_priority"].to_dict()
    weights = np.array([priorities.get(i, 0.0) for i in range(proba.shape[1])])
    score = proba @ weights

    # Небольшая стабилизация через node_strength: не как отдельный экспертный score,
    # а как геологическая интерпретация компонент GMM.
    score = 0.82 * normalize01(score) + 0.18 * grid_local["node_strength"].values
    return normalize01(score)


def hitrate_at_quantile(grid_local, score, q=0.90):
    if grid_local["evidence_cell"].sum() == 0:
        return np.nan
    thr = np.quantile(score, q)
    top = score >= thr
    ev = grid_local["evidence_cell"].values.astype(bool)
    return float((top & ev).sum() / ev.sum())


def mean_node_top(grid_local, score, q=0.90):
    thr = np.quantile(score, q)
    return float(grid_local.loc[score >= thr, "node_strength"].mean())

results = []
cluster_tables = []
best = None

for n_comp in N_COMPONENTS_LIST:
    for cov in COVARIANCE_TYPES:
        name = f"GMM_k{n_comp}_{cov}"
        try:
            model = GaussianMixture(
                n_components=n_comp,
                covariance_type=cov,
                reg_covar=1e-4,
                n_init=8,
                max_iter=500,
                random_state=RANDOM_STATE,
            )
            labels = model.fit_predict(X)
            proba = model.predict_proba(X)
            stat = cluster_statistics(grid, labels, proba, model_name=name)
            score = prospectivity_from_gmm(grid, labels, proba, stat)

            h10 = hitrate_at_quantile(grid, score, q=0.90)
            h15 = hitrate_at_quantile(grid, score, q=0.85)
            node10 = mean_node_top(grid, score, q=0.90)
            bic = model.bic(X)
            aic = model.aic(X)

            # objective: evidence + node + небольшой штраф за слишком сложные модели по BIC
            # BIC используем только мягко, чтобы не выбирать без необходимости слишком сложную модель.
            results.append({
                "model_name": name,
                "n_components": n_comp,
                "covariance_type": cov,
                "hitrate_top10": h10,
                "hitrate_top15": h15,
                "mean_node_top10": node10,
                "bic": float(bic),
                "aic": float(aic),
                "converged": bool(model.converged_),
            })
            cluster_tables.append(stat)
        except Exception as e:
            print("Ошибка:", name, "->", e)

res = pd.DataFrame(results)

# Нормируем BIC: меньше BIC лучше
if len(res) == 0:
    raise RuntimeError("Не удалось обучить ни одну GMM-конфигурацию")

bic_good = 1.0 - normalize01(res["bic"].values)

# Если evidence нет, h10/h15 будут NaN — тогда выбираем по node_top10 и BIC
h10 = res["hitrate_top10"].fillna(0).values
h15 = res["hitrate_top15"].fillna(0).values
node10 = res["mean_node_top10"].fillna(0).values

res["objective"] = (
    0.42 * normalize01(h10) +
    0.23 * normalize01(h15) +
    0.25 * normalize01(node10) +
    0.10 * bic_good
)

res = res.sort_values("objective", ascending=False).reset_index(drop=True)
display(res)

best_row = res.iloc[0]
print("Лучшая конфигурация:")
print(best_row)

In [ ]:
# ============================================================
# 8. ФИНАЛЬНАЯ GMM-МОДЕЛЬ
# ============================================================

best_n = int(best_row["n_components"])
best_cov = str(best_row["covariance_type"])

final_model = GaussianMixture(
    n_components=best_n,
    covariance_type=best_cov,
    reg_covar=1e-4,
    n_init=12,
    max_iter=700,
    random_state=RANDOM_STATE,
)

final_labels = final_model.fit_predict(X)
final_proba = final_model.predict_proba(X)
final_stat = cluster_statistics(grid, final_labels, final_proba, model_name=f"FINAL_GMM_k{best_n}_{best_cov}")
final_score = prospectivity_from_gmm(grid, final_labels, final_proba, final_stat)

grid["gmm_cluster"] = final_labels
grid["prospectivity"] = normalize01(final_score)
grid["prognoz"] = 1.0 - grid["prospectivity"]

print("Финальная модель:", f"GMM_k{best_n}_{best_cov}")
print("Converged:", final_model.converged_)
print("Evidence cells:", int(grid["evidence_cell"].sum()))
print("HitRate top 10%:", hitrate_at_quantile(grid, grid["prospectivity"].values, q=0.90))
print("HitRate top 15%:", hitrate_at_quantile(grid, grid["prospectivity"].values, q=0.85))

final_stat = final_stat.sort_values("cluster_priority", ascending=False).reset_index(drop=True)
display(final_stat)
final_stat.to_csv(CLUSTER_TABLE_PATH, index=False, encoding="utf-8-sig")
print("Таблица компонент сохранена:", CLUSTER_TABLE_PATH)

In [ ]:
# ============================================================
# 9. СГЛАЖИВАНИЕ И ВЫДЕЛЕНИЕ КОМПАКТНЫХ TOP-ЗОН
# ============================================================

def make_grid_array(grid_local, value_col, fill=np.nan):
    n_rows = int(grid_local["row"].max()) + 1
    n_cols = int(grid_local["col"].max()) + 1
    arr = np.full((n_rows, n_cols), fill, dtype=float)
    mask_arr = np.zeros((n_rows, n_cols), dtype=bool)
    rr = grid_local["row"].astype(int).values
    cc = grid_local["col"].astype(int).values
    arr[rr, cc] = grid_local[value_col].values
    mask_arr[rr, cc] = True
    return arr, mask_arr


def smooth_grid_score(grid_local, value_col="prospectivity", sigma=0.9):
    arr, mask_arr = make_grid_array(grid_local, value_col)
    if not SCIPY_OK or sigma <= 0:
        out = arr.copy()
    else:
        val = np.where(mask_arr, arr, 0.0)
        w = mask_arr.astype(float)
        sm_val = gaussian_filter(val, sigma=sigma)
        sm_w = gaussian_filter(w, sigma=sigma)
        out = np.divide(sm_val, sm_w, out=np.zeros_like(sm_val), where=sm_w > 1e-9)
        out[~mask_arr] = np.nan
    rr = grid_local["row"].astype(int).values
    cc = grid_local["col"].astype(int).values
    return normalize01(out[rr, cc])


def connected_component_filter(grid_local, mask_values, min_cells=3, max_cells=250):
    """Фильтр связных компонент на регулярной сетке."""
    n_rows = int(grid_local["row"].max()) + 1
    n_cols = int(grid_local["col"].max()) + 1
    arr = np.zeros((n_rows, n_cols), dtype=bool)
    rr = grid_local["row"].astype(int).values
    cc = grid_local["col"].astype(int).values
    arr[rr, cc] = mask_values.astype(bool)

    if SCIPY_OK:
        structure = np.array([[0,1,0], [1,1,1], [0,1,0]], dtype=int)
        lab, nlab = cc_label(arr, structure=structure)
        keep = np.zeros_like(arr, dtype=bool)
        for k in range(1, nlab + 1):
            size = int((lab == k).sum())
            if min_cells <= size <= max_cells:
                keep[lab == k] = True
        return keep[rr, cc]

    # fallback без scipy: если scipy нет, возвращаем исходную маску
    return mask_values.astype(bool)


grid["prospectivity_smooth"] = smooth_grid_score(grid, "prospectivity", sigma=SMOOTH_SIGMA)

thr = float(grid["prospectivity_smooth"].quantile(TOP_ZONE_QUANTILE))
raw_top = grid["prospectivity_smooth"].values >= thr

# Дополнительное условие: top-зоны должны быть связаны с узловыми признаками
node_thr = float(grid["node_strength"].quantile(0.75))
raw_top = raw_top & (grid["node_strength"].values >= node_thr)

top_filtered = connected_component_filter(grid, raw_top, min_cells=2, max_cells=220)

# Если фильтр получился слишком строгим, оставляем просто top по квантилю
if top_filtered.sum() < 10:
    print("Top-zone фильтр получился слишком строгим, использую более мягкий порог.")
    thr = float(grid["prospectivity_smooth"].quantile(0.990))
    raw_top = (grid["prospectivity_smooth"].values >= thr) & (grid["node_strength"].values >= grid["node_strength"].quantile(0.65))
    top_filtered = connected_component_filter(grid, raw_top, min_cells=2, max_cells=260)

grid["top_zone"] = top_filtered.astype(int)

ev_total = int(grid["evidence_cell"].sum())
ev_top = int(((grid["evidence_cell"] == 1) & (grid["top_zone"] == 1)).sum())

metrics = {
    "method": "GaussianMixture_unsupervised",
    "n_cells": int(len(grid)),
    "n_features": int(len(FEATURE_COLS)),
    "best_n_components": best_n,
    "best_covariance_type": best_cov,
    "evidence_cells": ev_total,
    "top_zone_cells": int(grid["top_zone"].sum()),
    "top_zone_share": float(grid["top_zone"].mean()),
    "evidence_in_top_zone": ev_top,
    "hitrate_top_zone": float(ev_top / ev_total) if ev_total > 0 else None,
    "hitrate_top10": hitrate_at_quantile(grid, grid["prospectivity_smooth"].values, q=0.90),
    "hitrate_top15": hitrate_at_quantile(grid, grid["prospectivity_smooth"].values, q=0.85),
    "mean_node_top_zone": float(grid.loc[grid["top_zone"] == 1, "node_strength"].mean()) if grid["top_zone"].sum() > 0 else None,
}

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print(json.dumps(metrics, ensure_ascii=False, indent=2))
print("Метрики сохранены:", METRICS_PATH)

In [ ]:
# ============================================================
# 10. ИТОГОВАЯ КАРТА
# ============================================================

fig, ax = plt.subplots(figsize=(11, 10))

# Синий = более перспективно, красный = менее перспективно
grid.plot(
    column="prospectivity_smooth",
    ax=ax,
    cmap="RdYlBu",
    linewidth=0,
    legend=True,
    legend_kwds={"label": "Относительная перспективность GMM"},
)

# Граница территории
mask.boundary.plot(ax=ax, color="black", linewidth=0.8)

# Top-зоны желтым
if grid["top_zone"].sum() > 0:
    grid.loc[grid["top_zone"] == 1].boundary.plot(ax=ax, color="gold", linewidth=1.3)
    grid.loc[grid["top_zone"] == 1].plot(ax=ax, color="gold", alpha=0.35, linewidth=0)

# Evidence points
if len(evidence) > 0:
    evidence.plot(ax=ax, color="yellow", edgecolor="black", markersize=20, linewidth=0.4, alpha=0.85)

ax.set_title(
    f"Gaussian Mixture Model: карта относительной перспективности\n"
    f"GMM k={best_n}, covariance={best_cov}; top-zone={grid['top_zone'].mean()*100:.2f}%",
    fontsize=13
)
ax.set_axis_off()
plt.tight_layout()
plt.savefig(PNG_PATH, dpi=300, bbox_inches="tight")
plt.show()

print("PNG сохранен:", PNG_PATH)

## Как интерпретировать результат

- `prospectivity_smooth` — относительная перспективность по GMM после сглаживания.
- `prognoz = 1 - prospectivity` — обратная шкала, если нужна логика «меньше значение — перспективнее».
- `gmm_cluster` — скрытый геологический тип территории.
- `top_zone` — самые компактные перспективные участки.

Важно: GMM не обучается на `evidence_cell` как классификатор. Известные проявления используются для интерпретации компонент и оценки результата.